# Program *Training*: Jaringan Saraf Tiruan dengan *Back Propagation*

Mengenali 3 kelas kendaraan (pesawat, mobil, kapal) dari citra *grayscale* (abu-abu, satu nilai terang tiap *pixel*) 32 x 32.
Algoritma dan notasi mengikuti Fausett, *Fundamentals of Neural Networks* (1994), Bab 6, Langkah 0 sampai 9.

Hasil akhir: satu berkas `BOBOT.npz` berisi `v`, `v_0`, `w`, `w_0`.

## Sel 1. Pengaturan Awal dan Konfigurasi Jaringan

* Menyambungkan Google Colab ke Google Drive serta menetapkan alamat folder (*path*) *dataset* dan berkas luaran bobot (`BOBOT.npz`).
* Menetapkan arsitektur jaringan: 1024 *node input* (citra $32 \times 32$), 100 *node hidden* (lapisan tengah), dan 3 *node output* (pesawat, mobil, kapal).
* Mengatur *hyperparameter* (pengaturan yang ditetapkan sebelum *training*):
   * *learning rate* (`alpha` = 0.1): besar langkah koreksi bobot tiap pembaruan.
   * *epoch* maksimum 200 (*epoch*: satu putaran penuh melewati seluruh data latih).
   * ambang galat 0.01 (galat: selisih keluaran jaringan dan target).
   * rentang bobot awal acak dan *random seed* (angka awal pengacak agar hasil bisa diulang).
* **Analogi:** seperti menyiapkan ruang kelas sebelum pelajaran: menentukan ukuran kertas soal, seberapa besar koreksi tiap kesalahan (*learning rate*), dan kapan latihan berhenti.

In [ ]:
import os                      # akses folder dan berkas
import time                    # mengukur lama training
import numpy as np             # operasi matriks dan vektor
from PIL import Image          # membaca gambar (Pillow)
import matplotlib.pyplot as plt  # menggambar grafik dan gambar
from google.colab import drive   # menyambungkan Colab ke Google Drive

drive.mount('/content/drive')  # sambungkan Colab ke Google Drive

# Lokasi di Google Drive
FOLDER_PROYEK = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI'
FOLDER_LATIH  = FOLDER_PROYEK + '/DATASET/LATIH'   # gambar untuk belajar
FOLDER_UJI    = FOLDER_PROYEK + '/DATASET/UJI'     # gambar untuk menguji (tidak ikut dilatih)
BERKAS_BOBOT  = FOLDER_PROYEK + '/BOBOT/BOBOT.npz' # tempat menyimpan hasil training

# Ukuran gambar dan arsitektur
LEBAR, TINGGI = 32, 32  # ukuran gambar dalam pixel (titik gambar)
n = LEBAR * TINGGI      # node input: satu node untuk tiap pixel
p = 100                 # node hidden: lapisan tengah pengolah pola
m = 3                   # node output: satu per kelas (harus tepat 3)

# Parameter pembelajaran
alpha        = 0.1      # learning rate: besar langkah koreksi bobot tiap pembaruan
EPOCH_MAKS   = 200      # epoch: satu putaran penuh seluruh data latih; ini batas maksimum (syarat berhenti 1)
AMBANG_GALAT = 0.01     # galat: selisih keluaran dan target; berhenti bila rata-rata per pola di bawah ini (syarat berhenti 2)
BATAS_ACAK   = 0.5      # bobot awal diacak antara -0,5 dan 0,5
BIJI_ACAK    = 42       # random seed: angka awal pengacak agar hasil bisa diulang
CETAK_TIAP   = 10       # interval laporan: cetak hasil tiap 10 epoch

# Nama folder tiap kelas. Urutannya menentukan target: indeks 0 = 1 0 0, indeks 1 = 0 1 0, indeks 2 = 0 0 1
NAMA_KELAS = ['KELAS1_PESAWAT', 'KELAS2_MOBIL', 'KELAS3_KAPAL']

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Node input  (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p) : {p}')
print(f'Node output (m) : {m}')
print(f'alpha           : {alpha}')
print(f'Epoch maksimum  : {EPOCH_MAKS}')
print(f'Ambang galat    : {AMBANG_GALAT}')

## Sel 2. Prapemrosesan Citra dan Pemuatan *Dataset*

* Fungsi `praproses` mengubah satu gambar menjadi vektor masukan dalam 4 langkah:
   * *grayscale*: ubah warna menjadi satu nilai terang tiap *pixel*.
   * pengaman ukuran: bila ukuran bukan $32 \times 32$, ubah ukurannya (*resize*).
   * *flatten*: ubah gambar 2 dimensi menjadi satu baris berisi 1024 nilai.
   * normalisasi: bagi dengan 255.0 agar nilai berada pada skala 0 hingga 1.
* Fungsi `muat_dataset` memuat seluruh data latih dan uji, lalu memetakan label target ke format *one-hot* (hanya satu *node* bernilai 1, sisanya 0): `1 0 0`, `0 1 0`, atau `0 0 1`.
* **Analogi:** seperti menyalin semua kartu soal ke kertas hitam putih berukuran sama, lalu mencatat kunci jawabannya dari nama folder tempat kartu itu disimpan.

In [ ]:
def praproses(gambar):
    """Ubah satu gambar PIL (RGB) menjadi vektor x berisi n nilai, rentang 0 sampai 1."""
    abu = gambar.convert('L')                       # 1. grayscale: warna jadi satu nilai terang per pixel
    if abu.size != (LEBAR, TINGGI):                 # 2. pengaman ukuran: bila ukuran beda,
        abu = abu.resize((LEBAR, TINGGI))           #    ubah ukuran (resize) ke 32 x 32
    x = np.asarray(abu, dtype=np.float64).flatten() # 3. flatten: ubah gambar 2 dimensi jadi satu baris
    return x / 255.0                                # 4. normalisasi: bagi 255 agar nilai 0 sampai 1


def muat_dataset(folder):
    """Baca semua gambar di folder. Hasil: X (data), T (target), label (nomor kelas), nama berkas."""
    X, T, label, nama_berkas = [], [], [], []
    for k, nama in enumerate(NAMA_KELAS):
        # sorted: urutkan nama berkas agar urutan baca selalu sama
        daftar = sorted(f for f in os.listdir(f'{folder}/{nama}') if f.endswith('.png'))
        for berkas in daftar:
            gambar = Image.open(f'{folder}/{nama}/{berkas}')
            X.append(praproses(gambar))
            target = np.zeros(m)
            target[k] = 1.0                         # one-hot: hanya node kelas ini bernilai 1 (100, 010, atau 001)
            T.append(target)
            label.append(k)                         # label: nomor kelas (0, 1, atau 2), dipakai menghitung akurasi
            nama_berkas.append(f'{nama}/{berkas}')
    return np.array(X), np.array(T), np.array(label), nama_berkas


X_latih, T_latih, label_latih, nama_latih = muat_dataset(FOLDER_LATIH)   # nama_latih: nama berkas tiap gambar latih
X_uji,   T_uji,   label_uji,   _ = muat_dataset(FOLDER_UJI)

print('=' * 52)
print('DATASET DIMUAT')
print('=' * 52)
print(f'{"Kelas":<18}{"Latih":>8}{"Uji":>8}   Target')
for k, nama in enumerate(NAMA_KELAS):
    target = ' '.join(str(int(a)) for a in np.eye(m)[k])
    print(f'{nama:<18}{(label_latih == k).sum():>8}{(label_uji == k).sum():>8}   {target}')
print('-' * 52)
print(f'{"Total":<18}{len(X_latih):>8}{len(X_uji):>8}')
print(f'\nBentuk X_latih : {X_latih.shape}  (pola x node input)')
print(f'Rentang nilai  : {X_latih.min():.2f} sampai {X_latih.max():.2f}')

## Sel 3. Verifikasi Konversi Citra

* Menampilkan perbandingan visual antara gambar asli berwarna di baris atas dengan hasil olah *grayscale* di baris bawah untuk memvalidasi masukan model.
* **Analogi:** seperti memeriksa hasil salinan sebelum dipakai: bentuk gambar masih sama, hanya warnanya yang hilang.

In [ ]:
# baris atas: gambar asli berwarna; baris bawah: hasil praproses (grayscale)
fig, sumbu = plt.subplots(2, 3, figsize=(7, 5))
for k, nama in enumerate(NAMA_KELAS):
    # ambil gambar pertama tiap kelas sebagai contoh
    berkas = sorted(f for f in os.listdir(f'{FOLDER_LATIH}/{nama}') if f.endswith('.png'))[0]
    asli = Image.open(f'{FOLDER_LATIH}/{nama}/{berkas}')
    sumbu[0, k].imshow(asli)
    sumbu[0, k].set_title(nama.split('_')[1] + f'\n(mode {asli.mode})')
    sumbu[1, k].imshow(praproses(asli).reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1)
    sumbu[1, k].set_title('grayscale')
    sumbu[0, k].axis('off'); sumbu[1, k].axis('off')
plt.tight_layout()
plt.show()

## Sel 4. Fungsi Aktivasi dan Inisialisasi Bobot Awal (Step 0)

* Menyiapkan fungsi aktivasi sigmoid `f(x) = 1 / (1 + e^-x)` (mengubah nilai apa pun menjadi 0 sampai 1) dan turunannya `f'(x) = f(x)(1 - f(x))`.
* Mengisi seluruh bobot dan *bias* (nilai tambahan tetap tiap *node*) secara acak seragam antara $-0.5$ hingga $0.5$ sebagai tebakan mula-mula model.
* *Output* sel diberi label **Step 0** (*Initialize weights*) seperti di buku, disertai contoh beberapa nilai bobot awal.
* **Analogi:** sigmoid seperti tombol peredup lampu: sekuat apa pun dorongannya, terangnya hanya antara 0 (mati) dan 1 (terang penuh). Bobot acak seperti murid yang menebak asal sebelum mulai belajar.

In [ ]:
def f(x):
    # fungsi aktivasi sigmoid: ubah nilai apa pun menjadi 0 sampai 1
    return 1.0 / (1.0 + np.exp(-x))

def f_aksen(x):
    # turunan sigmoid: dipakai menghitung koreksi galat
    return f(x) * (1.0 - f(x))


# Langkah 0: inisialisasi bobot
rng = np.random.default_rng(BIJI_ACAK)               # rng: pembangkit bilangan acak (memakai random seed di atas)
v   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (n, p))   # bobot V: dari node input ke node hidden
v_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, p)        # bias node hidden: nilai tambahan tetap tiap node hidden
w   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (p, m))   # bobot W: dari node hidden ke node output
w_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, m)        # bias node output: nilai tambahan tetap tiap node output


print('=' * 62)
print('Step 0 : Initialize weights (set to small random values)')
print('=' * 62)
print(f'{"Variabel":<10}{"Ukuran":<14}{"Jumlah nilai":>12}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}{larik.size:>12,}')
print(f'\nRentang bobot awal: {min(v.min(), w.min()):.2f} sampai {max(v.max(), w.max()):.2f}')

# contoh nilai: indeks buku mulai dari 1, indeks Python mulai dari 0 (v_11 = v[0, 0])
print('\nContoh nilai bobot awal:')
print(f'  v_11 = {v[0, 0]:+.4f}   v_21 = {v[1, 0]:+.4f}   v_0,1 = {v_0[0]:+.4f}')
print(f'  w_11 = {w[0, 0]:+.4f}   w_21 = {w[1, 0]:+.4f}   w_0,1 = {w_0[0]:+.4f}')

## Sel 5. Siklus Pelatihan *Back Propagation* (Step 1 sampai 9)

* Semua rumus ditulis seperti di buku memakai *looping* (perulangan `for`), tanpa operator `@` dan tanpa `np.outer`. Contoh Step 4: `z_in_j = v_0j + SUM_i x_i v_ij` ditulis per *node hidden* j, dan jumlah atas i dikerjakan `np.sum`. Perulangan i satu per satu terlalu lambat di Python (sekitar 35 detik per *epoch* hanya untuk Step 4).
* Umpan maju (*feedforward*): Step 3 sampai 5. Propagasi galat mundur (*backpropagation of error*): Step 6 sampai 8. Syarat berhenti: Step 9.
* *Output* diberi label **Step 1** sampai **Step 9**. Rincian Step 3 sampai 8 (nilai nyata `z_in`, `z`, `y_in`, `y`, `delta_k`, `delta_j`, dan perubahan bobot) dicetak untuk pasangan latih pertama pada *epoch* 1. Setelah itu, tiap 10 *epoch* dicetak ringkasan Step 1, Step 2, dan Step 9.
* **Analogi:** seperti murid mengerjakan satu soal lalu dikoreksi guru: ia menjawab (umpan maju), melihat seberapa salah jawabannya (galat), lalu memperbaiki seberapa penting ia menimbang tiap petunjuk di gambar (bobot). Diulang untuk semua kartu, berkas demi berkas.

In [ ]:
def umpan_maju(X):
    """Step 4 dan 5 untuk tiap pola di X, satu per satu. Hasil: matriks y (baris = pola). Bobot tidak diubah."""
    hasil = np.zeros((len(X), m))
    for s in range(len(X)):
        x = X[s]
        z_in = np.zeros(p)                           # z_in: masukan bersih tiap node hidden
        z    = np.zeros(p)                           # z: keluaran tiap node hidden
        for j in range(p):                           # Step 4: tiap node hidden Z_j
            z_in[j] = v_0[j] + np.sum(x * v[:, j])   # z_in_j = v_0j + SUM_i x_i v_ij (np.sum: jumlahkan semua suku)
            z[j]    = f(z_in[j])                     # z_j = f(z_in_j)
        for k in range(m):                           # Step 5: tiap node output Y_k
            y_in_k = w_0[k] + np.sum(z * w[:, k])    # y_in_k = w_0k + SUM_j z_j w_jk
            hasil[s, k] = f(y_in_k)                  # y_k = f(y_in_k)
    return hasil

def akurasi(X, label):
    # argmax: nomor node output bernilai terbesar = kelas tebakan; akurasi = persen tebakan yang benar
    return np.mean(np.argmax(umpan_maju(X), axis=1) == label) * 100


riwayat_galat, riwayat_akurasi = [], []   # catatan galat dan akurasi tiap epoch, untuk grafik
jumlah_pola = len(X_latih)                # pola = satu gambar latih beserta targetnya

print('=' * 62)
print('Step 0 : Initialize weights (sudah dikerjakan di Sel 4)')
print('Step 1 : While stopping condition is false, do Steps 2 to 9')
print(f'         berhenti bila galat rata-rata < {AMBANG_GALAT} atau epoch mencapai {EPOCH_MAKS}')
print('=' * 62)

waktu_mulai = time.time()

# Step 1: ulangi selama syarat berhenti belum terpenuhi (satu putaran = satu epoch)
for epoch in range(1, EPOCH_MAKS + 1):
    tampil = (epoch == 1 or epoch % CETAK_TIAP == 0)   # tampil: apakah laporan epoch ini dicetak
    if tampil:
        print(f'\nEPOCH {epoch}')
        print('-' * 62)
        print('Step 1 : syarat berhenti belum terpenuhi, kerjakan Step 2 sampai 9')
        print(f'Step 2 : For each training pair, do Steps 3 to 8  ({jumlah_pola} pasangan, urutan diacak)')

    galat_total = 0.0                      # jumlah galat semua pola dalam epoch ini

    # Step 2: untuk setiap pasangan (x, t); urutan pola diacak tiap epoch (rng.permutation)
    # enumerate: memberi nomor urut 0, 1, 2, ... pada tiap pola
    for nomor, s in enumerate(rng.permutation(jumlah_pola)):
        x = X_latih[s]                     # x: masukan (nilai 1024 pixel)
        t = T_latih[s]                     # t: target (100, 010, atau 001)
        rincian = (epoch == 1 and nomor == 0)   # rincian: cetak nilai nyata Step 3 sampai 8 hanya untuk pasangan pertama

        if rincian:
            print(f'\n  Rincian pasangan pertama {nama_latih[s]}, target t = {" ".join(str(int(a)) for a in t)}')

        # Step 3: tiap node input X_i menerima x_i dan meneruskannya ke semua node hidden
        if rincian:
            print('  Step 3 : input unit X_i menerima x_i, lalu dikirim ke semua hidden unit')
            print(f'           x_1 = {x[0]:.4f}   x_2 = {x[1]:.4f}   x_3 = {x[2]:.4f}   ... ({n} nilai)')

        # Step 4: tiap node hidden Z_j menjumlahkan sinyal masukan berbobot, lalu memakai fungsi aktivasi
        z_in = np.zeros(p)                           # z_in: masukan bersih tiap node hidden
        z    = np.zeros(p)                           # z: keluaran tiap node hidden
        for j in range(p):
            z_in[j] = v_0[j] + np.sum(x * v[:, j])   # z_in_j = v_0j + SUM_i x_i v_ij (np.sum: jumlahkan semua suku)
            z[j]    = f(z_in[j])                     # z_j = f(z_in_j)
        if rincian:
            print('  Step 4 : z_in_j = v_0j + SUM_i x_i v_ij ;  z_j = f(z_in_j)')
            for j in range(3):
                print(f'           j = {j + 1}: z_in = {z_in[j]:+.4f}   z = {z[j]:.4f}')
            print(f'           ... ({p} node hidden)')

        # Step 5: tiap node output Y_k menjumlahkan sinyal masukan berbobot, lalu memakai fungsi aktivasi
        y_in = np.zeros(m)                           # y_in: masukan bersih tiap node output
        y    = np.zeros(m)                           # y: keluaran tiap node output
        for k in range(m):
            y_in[k] = w_0[k] + np.sum(z * w[:, k])   # y_in_k = w_0k + SUM_j z_j w_jk
            y[k]    = f(y_in[k])                     # y_k = f(y_in_k)
        if rincian:
            print('  Step 5 : y_in_k = w_0k + SUM_j z_j w_jk ;  y_k = f(y_in_k)')
            for k in range(m):
                print(f'           k = {k + 1}: y_in = {y_in[k]:+.4f}   y = {y[k]:.4f}')

        # Step 6: tiap node output Y_k menerima target t_k, lalu menghitung faktor koreksi galat delta_k
        delta_k   = np.zeros(m)                      # delta_k: faktor koreksi galat node output
        delta_w_0 = np.zeros(m)                      # koreksi bias node output
        delta_w   = np.zeros((p, m))                 # koreksi bobot W
        for k in range(m):
            delta_k[k]   = (t[k] - y[k]) * f_aksen(y_in[k])    # delta_k = (t_k - y_k) f'(y_in_k)
            delta_w_0[k] = alpha * delta_k[k]                  # delta_w_0k = alpha delta_k
            for j in range(p):
                delta_w[j, k] = alpha * delta_k[k] * z[j]      # delta_w_jk = alpha delta_k z_j
        if rincian:
            print("  Step 6 : delta_k = (t_k - y_k) f'(y_in_k) ;  delta_w_jk = alpha delta_k z_j")
            for k in range(m):
                print(f'           k = {k + 1}: t = {t[k]:.0f}   y = {y[k]:.4f}   delta_k = {delta_k[k]:+.5f}')

        # Step 7: tiap node hidden Z_j menjumlahkan delta dari node output, lalu menghitung delta_j
        delta_in_j = np.zeros(p)                     # delta_in_j: jumlah delta masukan node hidden
        delta_j    = np.zeros(p)                     # delta_j: faktor koreksi galat node hidden
        delta_v_0  = np.zeros(p)                     # koreksi bias node hidden
        delta_v    = np.zeros((n, p))                # koreksi bobot V
        for j in range(p):
            delta_in_j[j] = np.sum(delta_k * w[j, :])           # delta_in_j = SUM_k delta_k w_jk (w masih nilai lama)
            delta_j[j]    = delta_in_j[j] * f_aksen(z_in[j])    # delta_j = delta_in_j f'(z_in_j)
            delta_v_0[j]  = alpha * delta_j[j]                  # delta_v_0j = alpha delta_j
            delta_v[:, j] = alpha * delta_j[j] * x              # delta_v_ij = alpha delta_j x_i (semua i sekaligus)
        if rincian:
            print("  Step 7 : delta_in_j = SUM_k delta_k w_jk ;  delta_j = delta_in_j f'(z_in_j)")
            for j in range(3):
                print(f'           j = {j + 1}: delta_in = {delta_in_j[j]:+.5f}   delta_j = {delta_j[j]:+.6f}')
            print(f'           ... ({p} node hidden)')
            w_lama = w[0, 0]                         # simpan w_11 dan v_11 lama untuk dicetak setelah Step 8
            v_lama = v[0, 0]

        # Step 8: bobot dan bias baru = bobot dan bias lama + koreksi
        for k in range(m):
            w_0[k]  = w_0[k] + delta_w_0[k]          # w_0k(new) = w_0k(old) + delta_w_0k
            w[:, k] = w[:, k] + delta_w[:, k]        # w_jk(new) = w_jk(old) + delta_w_jk (semua j sekaligus)
        for j in range(p):
            v_0[j]  = v_0[j] + delta_v_0[j]          # v_0j(new) = v_0j(old) + delta_v_0j
            v[:, j] = v[:, j] + delta_v[:, j]        # v_ij(new) = v_ij(old) + delta_v_ij (semua i sekaligus)
        if rincian:
            print('  Step 8 : w_jk(new) = w_jk(old) + delta_w_jk ;  v_ij(new) = v_ij(old) + delta_v_ij')
            print(f'           w_11 {w_lama:+.5f} menjadi {w[0, 0]:+.5f}')
            print(f'           v_11 {v_lama:+.5f} menjadi {v[0, 0]:+.5f}')
            print(f'           ({jumlah_pola - 1} pasangan lain memakai Step 3 sampai 8 yang sama, tanpa dicetak)')

        galat_total += 0.5 * np.sum((t - y) ** 2)    # galat satu pola: setengah jumlah kuadrat selisih target dan keluaran

    galat_rata = galat_total / jumlah_pola           # galat rata-rata per pola pada epoch ini
    riwayat_galat.append(galat_rata)
    riwayat_akurasi.append(akurasi(X_latih, label_latih))

    # Step 9: uji syarat berhenti (galat di bawah ambang, atau epoch maksimum tercapai)
    berhenti = (galat_rata < AMBANG_GALAT) or (epoch == EPOCH_MAKS)
    if tampil or berhenti:
        if not tampil:                               # epoch berhenti yang belum dicetak: beri judul dulu
            print(f'\nEPOCH {epoch}')
            print('-' * 62)
        print(f'Step 2 : selesai, {jumlah_pola} pasangan sudah diproses')
        print('Step 9 : Test stopping condition')
        print(f'         galat rata-rata = {galat_rata:.5f} (ambang {AMBANG_GALAT}), akurasi latih = {riwayat_akurasi[-1]:.1f}%')
        if galat_rata < AMBANG_GALAT:
            print(f'         galat di bawah ambang: syarat berhenti terpenuhi pada epoch {epoch}')
        elif epoch == EPOCH_MAKS:
            print(f'         epoch maksimum ({EPOCH_MAKS}) tercapai: syarat berhenti terpenuhi')
        else:
            print('         syarat berhenti belum terpenuhi, kembali ke Step 1')
        if epoch == 1:
            print(f'         (waktu 1 epoch: {time.time() - waktu_mulai:.1f} detik)')
    if berhenti:
        break

print('\n' + '=' * 62)
print('RINGKASAN TRAINING')
print('=' * 62)
print(f'Epoch dijalankan : {len(riwayat_galat)}')
print(f'Galat akhir      : {riwayat_galat[-1]:.5f}')
print(f'Akurasi latih    : {riwayat_akurasi[-1]:.1f}%')
print(f'Waktu total      : {time.time() - waktu_mulai:.1f} detik')

## Sel 6. Grafik Hasil Belajar

* Menggambar rekaman galat rata-rata dan akurasi data latih untuk setiap *epoch* dari pelatihan tadi, sehingga penurunan galat dan kenaikan akurasi dapat dilihat tanpa komputasi ulang.
* **Analogi:** seperti mencatat nilai latihan tiap minggu: bila belajarnya berjalan baik, kesalahan makin turun dan nilai makin naik.

In [ ]:
# kiri: galat tiap epoch (diharapkan turun); kanan: akurasi data latih (diharapkan naik)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(range(1, len(riwayat_galat) + 1), riwayat_galat)
a1.set_title('Galat rata-rata per epoch'); a1.set_xlabel('Epoch'); a1.set_ylabel('Galat'); a1.grid(alpha=0.3)
a2.plot(range(1, len(riwayat_akurasi) + 1), riwayat_akurasi, color='tab:green')
a2.set_title('Akurasi data latih (%)'); a2.set_xlabel('Epoch'); a2.set_ylim(0, 100); a2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Sel 7. Evaluasi Model dan *Confusion Matrix*

* Menguji performa model pada data latih dan data uji, lalu membandingkan keduanya untuk melihat indikasi *overfitting* (model hafal data latih: akurasi latih jauh di atas akurasi uji).
* Menyajikan tabel tebakan (*confusion matrix*): baris adalah kelas sebenarnya dan kolom adalah tebakan model, sehingga angka di diagonal adalah tebakan tepat dan angka di luar diagonal adalah salah tebak tiap kelas.
* **Analogi:** seperti membandingkan nilai latihan soal dengan nilai ujian soal baru. Bila latihan nyaris sempurna tetapi ujian jauh lebih rendah, murid itu hanya menghafal soal latihan (*overfitting*).

In [ ]:
def laporan(X, label, judul):
    tebakan = np.argmax(umpan_maju(X), axis=1)   # tebakan: nomor node output bernilai terbesar
    benar = np.mean(tebakan == label) * 100
    print(f'{judul}: {benar:.1f}%  ({int((tebakan == label).sum())} dari {len(label)} gambar benar)')
    return tebakan

print('=' * 52)
print('EVALUASI')
print('=' * 52)
laporan(X_latih, label_latih, 'Akurasi data latih')
tebakan_uji = laporan(X_uji, label_uji, 'Akurasi data uji  ')

# confusion matrix: baris = kelas sebenarnya, kolom = tebakan model
print('\nTabel tebakan pada data uji')
judul = [nama.split('_')[1] for nama in NAMA_KELAS]
print(f'{"Sebenarnya":<12}' + ''.join(f'{j:>9}' for j in judul) + f'{"Akurasi":>10}')
for k in range(m):
    baris = [int(np.sum((label_uji == k) & (tebakan_uji == j))) for j in range(m)]
    persen = baris[k] / np.sum(label_uji == k) * 100
    print(f'{judul[k]:<12}' + ''.join(f'{b:>9}' for b in baris) + f'{persen:>9.0f}%')

## Sel 8. Inspeksi Keluaran Jaringan terhadap Target

* Menampilkan perbandingan nilai keluaran jaringan ($Y_1, Y_2, Y_3$) terhadap target pada sampel data uji (3 gambar tiap kelas) untuk memeriksa ketegasan tebakan model: node yang benar seharusnya mendekati 1, sisanya mendekati 0.
* **Analogi:** seperti membaca lembar jawaban berisi tiga lampu: lampu paling terang adalah jawaban jaringan, dan jawaban benar bila sama dengan lampu pada kunci jawaban.

In [ ]:
print('=' * 62)
print('POLA TARGET OUTPUT (t)')
print('=' * 62)
print(f'{"Kelas":<18}{"Y1":>4}{"Y2":>4}{"Y3":>4}   Artinya')
for k in range(m):
    pola = ''.join(f'{int(a):>4}' for a in np.eye(m)[k])
    print(f'{NAMA_KELAS[k]:<18}{pola}   hanya Y{k + 1} bernilai 1')

print()
print('=' * 62)
print('KELUARAN JARINGAN (y) DIBANDINGKAN DENGAN TARGET (t), DATA UJI')
print('=' * 62)
print(f'{"Kelas":<16}{"Target t":<10}{"Keluaran y (Y1, Y2, Y3)":<26}{"Hasil"}')
print('-' * 62)
y_uji = umpan_maju(X_uji)   # keluaran jaringan (y) untuk seluruh data uji
for k in range(m):
    for s in np.where(label_uji == k)[0][:3]:
        t_teks = ' '.join(str(int(a)) for a in T_uji[s])
        y_teks = '  '.join(f'{a:.3f}' for a in y_uji[s])
        # BENAR bila node dengan nilai terbesar adalah node kelas sebenarnya
        hasil = 'BENAR' if np.argmax(y_uji[s]) == k else 'SALAH'
        print(f'{NAMA_KELAS[k].split('_')[1]:<16}{t_teks:<10}{y_teks:<26}{hasil}')

## Sel 9. Penyimpanan Parameter Bobot Terlatih

* Menyimpan matriks bobot final ($v, v_0, w, w_0$) beserta ukuran gambar ke dalam satu berkas `BOBOT.npz` (format NumPy untuk beberapa larik dalam satu berkas) di Google Drive, lalu memverifikasi bahwa isi berkas sama dengan bobot di memori.
* **Analogi:** seperti menulis hasil belajar ke satu buku catatan, supaya saat ujian murid tinggal membaca catatan itu tanpa belajar dari awal.

In [ ]:
os.makedirs(os.path.dirname(BERKAS_BOBOT), exist_ok=True)   # buat folder BOBOT bila belum ada
# savez: simpan beberapa larik NumPy dalam satu berkas .npz
# ukuran gambar ikut disimpan agar program pengujian memakai ukuran yang sama
np.savez(BERKAS_BOBOT, v=v, v_0=v_0, w=w, w_0=w_0, ukuran=np.array([LEBAR, TINGGI]))

# Pemeriksaan: baca ulang berkas yang baru disimpan
cek = np.load(BERKAS_BOBOT)
print('=' * 52)
print('BOBOT TERSIMPAN')
print('=' * 52)
print(f'Lokasi : {BERKAS_BOBOT}')
print(f'Ukuran : {os.path.getsize(BERKAS_BOBOT) / 1024:.0f} KB\n')
print(f'{"Isi":<10}{"Ukuran":<14}')
for kunci in ['v', 'v_0', 'w', 'w_0']:
    print(f'{kunci:<10}{str(cek[kunci].shape):<14}')
sama = all(np.array_equal(cek[k], a) for k, a in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)])
print(f'\nIsi berkas sama dengan bobot di memori: {"YA" if sama else "TIDAK"}')